# Duke Breast Cancer MRI — 3D Bounding Box Detection (v4 — Bug 3 Fixed)
**Multimodal: 3D MRI volume + Clinical features + Radiomic imaging features**

## Fix 3 (this version) — Positional encoding on the slice dimension
Without positional encoding, `SliceAttention` treated all 32 depth slices as an
**unordered bag** — the model had no way to distinguish slice 5 from slice 25.
Z-localization was pure guessing, which capped val IoU at ~0.11 regardless of
XY accuracy because 3D IoU = IoU_xy × IoU_z.

**What changed:**
- `SliceAttention` now holds a learnable `nn.Embedding(max_depth, feat_dim)`.  
  Positional vectors are added to slice features once and shared across both
  `SliceAttention` and `CrossAttention`, so position flows into every attention
  computation.
- Embeddings are initialised with `std=0.02` (small) so training starts close
  to the v3 behaviour and adapts gradually.
- `max_depth` is set to 128 so the same checkpoint works if you increase DEPTH
  later without reinitialising.
- bbox_head bias init **removed** (it regressed P_016 from 0.504 → 0.218).
- PATIENCE raised to 20, NUM_EPOCHS to 120.

## All inherited fixes
- **Fix 1:** Augmentation flip decisions made in `__getitem__`, bbox label mirrored
  to match — image and label always consistent.
- **Fix 2:** Direct 6-output sigmoid head + 3D DIoU loss — breaks mean-box collapse.
- **v2 Bug 1:** Z-coordinate remapping via DICOM InstanceNumber range.
- **v2 Bug 2:** DICOMs sorted by InstanceNumber, not alphabetically.
- Series filtered to first post-contrast DCE phase (SeriesNumber 601).
- Pretrained ResNet18 2D backbone + cross-attention multimodal fusion.
- Warmup + cosine LR decay, separate backbone/head learning rates.

## Resolution upgrade path (when GPU allows)
After Bug 3 converges (~0.28–0.38 expected), the next lever is resolution:
```
DEPTH      = 64    # from 32  — 2× more Z context for pos encoding to exploit
IMAGE_SIZE = 192   # from 128 — finer XY detail for small tumors
BATCH_SIZE = 2     # may need to reduce for memory
```

In [1]:
import sys
!{sys.executable} -m pip install torch torchvision opencv-python matplotlib scikit-learn pillow pydicom pynrrd --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json, math, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch: 2.10.0+cu128
CUDA available: True


## 1. Inline Ground Truth Check

In [3]:
def _read_instance_number(path):
    try:
        dcm = pydicom.dcmread(str(path), stop_before_pixels=True)
        return int(getattr(dcm, 'InstanceNumber', 999999))
    except Exception:
        return 999999


def quick_gt_check(export_dir='exported_patients', max_patients=None):
    export_dir = Path(export_dir)
    for split in ('train', 'test'):
        split_dir = export_dir / split
        if not split_dir.exists():
            continue
        patient_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
        if max_patients:
            patient_dirs = patient_dirs[:max_patients]
        sort_bugs = z_bugs = z_ok = 0
        for pd_ in patient_dirs:
            data_file = pd_ / 'patient_data.json'
            mri_dir   = pd_ / 'MRI_DICOM_sample'
            if not data_file.exists() or not mri_dir.exists():
                continue
            with open(data_file) as f:
                meta = json.load(f)
            anns = meta.get('annotations', [])
            if not anns:
                continue
            bbox = anns[0].get('bounding_box_3d', {})
            z_min_gt = bbox.get('start_slice')
            z_max_gt = bbox.get('end_slice')
            if z_min_gt is None:
                continue
            dcm_paths = list(mri_dir.glob('**/*.dcm'))
            if not dcm_paths:
                continue
            instances = [(_read_instance_number(p), p) for p in dcm_paths]
            inst_sorted  = [p for _, p in sorted(instances, key=lambda x: x[0])]
            alpha_sorted = sorted(dcm_paths)
            if inst_sorted != alpha_sorted:
                sort_bugs += 1
            valid_inst = [i for i, _ in instances if i >= 0]
            if not valid_inst:
                continue
            inst_min, inst_max = min(valid_inst), max(valid_inst)
            overlap  = max(0, min(inst_max, z_max_gt) - max(inst_min, z_min_gt) + 1)
            gt_span  = max(z_max_gt - z_min_gt + 1, 1)
            coverage = 100.0 * overlap / gt_span
            if coverage < 50.0:
                z_bugs += 1
            else:
                z_ok += 1
        print(f'\n[{split.upper()}] {len(patient_dirs)} patients checked')
        print(f'  Sort mismatch : {sort_bugs}')
        print(f'  Z not covered : {z_bugs}')
        print(f'  Z OK          : {z_ok}')


quick_gt_check(export_dir='exported_patients')


[TRAIN] 730 patients checked
  Sort mismatch : 727
  Z not covered : 0
  Z OK          : 727

[TEST] 178 patients checked
  Sort mismatch : 175
  Z not covered : 0
  Z OK          : 175


## 2. Dataset (v3 — Fix 1 applied, unchanged)

In [4]:
class Duke3DBBoxDatasetV3(Dataset):
    """
    Fix 1: augmentation flip decisions made in __getitem__ and passed
    explicitly to _build_volume. Bbox label mirrored to match every flip.
    """

    CLINICAL_KEYS = [
        "Age at last contact in EMR f/u(days)(from the date of diagnosis) ,last time patient known to be alive, unless age of death is reported(in such case the age of death",
        "Menopause (at diagnosis)",
        "Days to MRI (From the Date of Diagnosis)",
        "Field Strength (Tesla)",
        "TR (Repetition Time)",
        "TE (Echo Time)",
        "Slice Thickness ",
        "Staging(Nodes)#(Nx replaced by -1)[N]",
        "Staging(Metastasis)#(Mx -replaced by -1)[M]",
        "Staging(Tumor Size)# [T]",
    ]

    IMAGING_KEYS = [
        "TumorMajorAxisLength_mm",
        "Volume_cu_mm_Tumor",
        "Energy_Tumor",
        "Contrast_Tumor",
        "Homogeneity1_Tumor",
        "Max_Enhancement_from_char_curv",
        "Time_to_Peak_from_char_curv",
        "Uptake_rate_from_char_curv",
        "Washout_rate_from_char_curv",
        "breastDensity_T1",
        "breastDensity_PostCon",
        "Peak_SER_tumor",
        "Median_Elongation_Tumor",
    ]

    def __init__(
        self, patients_dir, depth=32, image_size=128, augment=False,
        clinical_median=None, clinical_mean=None, clinical_std=None,
        imaging_median=None, imaging_mean=None, imaging_std=None,
    ):
        self.patients_dir = Path(patients_dir)
        self.depth = depth
        self.image_size = image_size
        self.augment = augment
        self.samples = []
        self.clinical_dim = len(self.CLINICAL_KEYS) + 4
        self.imaging_dim  = len(self.IMAGING_KEYS)
        self.clinical_median = clinical_median
        self.clinical_mean   = clinical_mean
        self.clinical_std    = clinical_std
        self.imaging_median  = imaging_median
        self.imaging_mean    = imaging_mean
        self.imaging_std     = imaging_std
        self._load_dataset()

    @staticmethod
    def _read_dicom_meta(path):
        try:
            dcm = pydicom.dcmread(str(path), stop_before_pixels=True)
            return (
                int(getattr(dcm, 'InstanceNumber',    999999)),
                str(getattr(dcm, 'SeriesInstanceUID', '')),
                int(getattr(dcm, 'SeriesNumber',      0)),
                int(getattr(dcm, 'Rows',              512)),
                int(getattr(dcm, 'Columns',           512)),
            )
        except Exception:
            return 999999, '', 0, 512, 512

    @staticmethod
    def _sort_by_instance(paths):
        """
        Filter to ONE series then sort by InstanceNumber.
        Prefers SeriesNumber 601 (first post-contrast, peak tumour enhancement).
        Falls back to most slices + highest SeriesNumber.
        """
        series = defaultdict(list)
        for p in paths:
            inst, uid, snum, rows, cols = Duke3DBBoxDatasetV3._read_dicom_meta(p)
            series[(snum, uid)].append((inst, p, rows, cols))
        if not series:
            return [], [], 512, 512
        best_key = None
        for key in series:
            snum, _ = key
            if snum == 601 and len(series[key]) > 5:
                best_key = key
                break
        if best_key is None:
            best_key = max(series.keys(), key=lambda k: (len(series[k]), k[0]))
        best = series[best_key]
        best.sort(key=lambda x: x[0])
        paths_out = [p    for _, p,    _, _ in best]
        insts_out = [inst for inst, _, _, _ in best]
        rows = best[0][2]
        cols = best[0][3]
        return paths_out, insts_out, rows, cols

    def _load_dataset(self):
        patient_folders = sorted([d for d in self.patients_dir.iterdir()
                           if d.is_dir() and not d.name.startswith('.')])
        raw_clinical, raw_imaging = [], []
        valid_samples = 0
        for patient_dir in patient_folders:
            data_file = patient_dir / "patient_data.json"
            if not data_file.exists():
                continue
            try:
                with open(data_file) as f:
                    metadata = json.load(f)
                mri_dir = patient_dir / "MRI_DICOM_sample"
                if not mri_dir.exists():
                    continue
                all_dcm = list(mri_dir.glob("**/*.dcm"))
                if not all_dcm:
                    continue
                sorted_dcm, instance_nums, img_rows, img_cols = self._sort_by_instance(all_dcm)
                annotations = metadata.get("annotations", [])
                if not annotations:
                    continue
                bbox_3d = annotations[0].get("bounding_box_3d", {})
                if not isinstance(bbox_3d, dict):
                    continue
                if bbox_3d.get("end_column", 0) <= bbox_3d.get("start_column", 0):
                    continue
                if bbox_3d.get("end_row", 0) <= bbox_3d.get("start_row", 0):
                    continue
                clinical = self._extract_clinical(metadata)
                imaging  = self._extract_imaging(metadata)
                self.samples.append(dict(
                    patient_id=patient_dir.name,
                    dicom_files=sorted_dcm,
                    instance_nums=instance_nums,
                    img_rows=img_rows,
                    img_cols=img_cols,
                    clinical_raw=clinical,
                    imaging_raw=imaging,
                    bbox_3d=bbox_3d,
                ))
                valid_samples += 1
                raw_clinical.append(clinical)
                raw_imaging.append(imaging)
            except Exception as e:
                print(f"  SKIP {patient_dir.name}: {type(e).__name__}: {e}")
        if self.clinical_mean is None and raw_clinical:
            arr = np.array(raw_clinical, dtype=np.float32)
            self.clinical_median = np.nanmedian(arr, axis=0)
            self.clinical_mean   = np.nanmean(arr, axis=0)
            self.clinical_std    = np.nanstd(arr, axis=0) + 1e-8
        if self.imaging_mean is None and raw_imaging:
            arr = np.array(raw_imaging, dtype=np.float32)
            self.imaging_median = np.nanmedian(arr, axis=0)
            self.imaging_mean   = np.nanmean(arr, axis=0)
            self.imaging_std    = np.nanstd(arr, axis=0) + 1e-8
        print(f"Dataset: {valid_samples} patients  "
              f"clinical_dim={self.clinical_dim}  imaging_dim={self.imaging_dim}")

    def _extract_clinical(self, metadata):
        clin = metadata.get("demographic_clinical", {})
        feats = []
        for key in self.CLINICAL_KEYS:
            val = clin.get(key, np.nan)
            try:
                v = float(val)
                feats.append(v if not np.isnan(v) else np.nan)
            except (TypeError, ValueError):
                feats.append(np.nan)
        grade = clin.get("Tumor Grade", 3)
        try:
            grade = int(float(grade))
        except (TypeError, ValueError):
            grade = 3
        grade = grade if grade in (1, 2, 3) else 3
        feats.append((grade - 1) / 2.0)
        for key in ("ER", "PR", "HER2"):
            try:
                feats.append(float(int(clin.get(key, 0))))
            except (TypeError, ValueError):
                feats.append(0.0)
        return np.array(feats, dtype=np.float32)

    def _extract_imaging(self, metadata):
        img = metadata.get("imaging_features", {})
        feats = []
        for key in self.IMAGING_KEYS:
            val = img.get(key, np.nan)
            try:
                v = float(val)
                feats.append(v if not np.isnan(v) else np.nan)
            except (TypeError, ValueError):
                feats.append(np.nan)
        return np.array(feats, dtype=np.float32)

    def _impute_normalise(self, arr, median, mean, std):
        result = arr.copy()
        nan_mask = np.isnan(result)
        if nan_mask.any() and median is not None:
            result[nan_mask] = median[nan_mask]
        result = np.nan_to_num(result, nan=0.0)
        return np.clip((result - mean) / std, -5.0, 5.0)

    def _load_dicom_pixels(self, path):
        try:
            dcm = pydicom.dcmread(str(path))
            img = dcm.pixel_array
            while len(img.shape) > 2 and img.shape[0] == 1:
                img = img.squeeze(0)
            if len(img.shape) == 3:
                img = img[img.shape[0] // 2] if img.shape[0] < img.shape[-1] else img[:, :, 0]
            if len(img.shape) != 2:
                return None
            return img.astype(np.float32)
        except Exception:
            return None

    def _build_volume(self, dicom_files, flip_x=False, flip_y=False, flip_z=False):
        """Fix 1: flip flags passed in from __getitem__ — no random decisions here."""
        n = len(dicom_files)
        if n == 0:
            return torch.zeros(1, self.depth, self.image_size, self.image_size)
        indices = np.linspace(0, n - 1, self.depth, dtype=int)
        slices = []
        for idx in indices:
            img = self._load_dicom_pixels(dicom_files[idx])
            if img is None:
                img = np.zeros((self.image_size, self.image_size), dtype=np.float32)
            else:
                img = np.array(
                    Image.fromarray(img).resize(
                        (self.image_size, self.image_size), Image.BILINEAR),
                    dtype=np.float32)
            slices.append(img)
        vol = np.stack(slices, axis=0)
        p1, p99 = np.percentile(vol, 1), np.percentile(vol, 99)
        if p99 > p1:
            vol = np.clip(vol, p1, p99)
        vol = (vol - vol.mean()) / (vol.std() + 1e-6)
        if flip_x:
            vol = vol[:, :, ::-1].copy()
        if flip_y:
            vol = vol[:, ::-1, :].copy()
        if flip_z:
            vol = vol[::-1, :, :].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                vol = vol * np.random.uniform(0.85, 1.15)
            if np.random.rand() < 0.3:
                vol = vol + np.random.normal(0, 0.02, vol.shape).astype(np.float32)
        return torch.tensor(vol, dtype=torch.float32).unsqueeze(0)

    def _create_bbox_target(self, bbox_3d, instance_nums, img_cols=512, img_rows=512):
        try:
            x_min = float(bbox_3d.get("start_column", 0))
            x_max = float(bbox_3d.get("end_column",   0))
            y_min = float(bbox_3d.get("start_row",    0))
            y_max = float(bbox_3d.get("end_row",      0))
            z_min_gt = bbox_3d.get("start_slice")
            if z_min_gt is None:
                z_min_gt = bbox_3d.get("start_image", 0)
            z_max_gt = bbox_3d.get("end_slice")
            if z_max_gt is None:
                z_max_gt = bbox_3d.get("end_image", 0)
            z_min_gt = float(z_min_gt)
            z_max_gt = float(z_max_gt)
            if x_max <= x_min or y_max <= y_min or z_max_gt <= z_min_gt:
                return torch.zeros(6, dtype=torch.float32)
            x_min_n = x_min / img_cols
            x_max_n = x_max / img_cols
            y_min_n = y_min / img_rows
            y_max_n = y_max / img_rows
            valid_inst = [i for i in instance_nums if 0 <= i < 999999]
            if not valid_inst:
                n = len(instance_nums)
                s_idx = np.linspace(0, n - 1, self.depth)
                zi = float(np.clip(np.searchsorted(s_idx, z_min_gt), 0, self.depth - 1))
                za = float(np.clip(np.searchsorted(s_idx, z_max_gt, side="right") - 1,
                                   0, self.depth - 1))
            else:
                inst_arr   = np.array(valid_inst)
                z_min_inst = z_min_gt + 1
                z_max_inst = z_max_gt + 1
                idx_min = float(np.searchsorted(inst_arr, z_min_inst, side="left"))
                idx_max = float(np.searchsorted(inst_arr, z_max_inst, side="right") - 1)
                n = len(inst_arr)
                zi = float(np.clip(idx_min * (self.depth - 1) / max(n - 1, 1), 0, self.depth - 1))
                za = float(np.clip(idx_max * (self.depth - 1) / max(n - 1, 1), 0, self.depth - 1))
            if za <= zi:
                za = min(zi + 1.0, self.depth - 1)
                zi = max(0.0, za - 1.0)
            return torch.clamp(torch.tensor([
                x_min_n, y_min_n, zi / (self.depth - 1),
                x_max_n, y_max_n, za / (self.depth - 1),
            ], dtype=torch.float32), 0.0, 1.0)
        except Exception:
            return torch.zeros(6, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Fix 1: decide flips here, pass explicitly
        flip_x = self.augment and (np.random.rand() < 0.5)
        flip_y = self.augment and (np.random.rand() < 0.5)
        flip_z = self.augment and (np.random.rand() < 0.5)
        volume = self._build_volume(
            s["dicom_files"],
            flip_x=flip_x, flip_y=flip_y, flip_z=flip_z)
        bbox = self._create_bbox_target(
            s["bbox_3d"], s["instance_nums"],
            img_cols=s.get("img_cols", 512),
            img_rows=s.get("img_rows", 512),
        )
        # Fix 1: mirror bbox coords to match flipped volume
        if flip_x:
            old_x1, old_x2 = bbox[0].clone(), bbox[3].clone()
            bbox[0] = 1.0 - old_x2
            bbox[3] = 1.0 - old_x1
        if flip_y:
            old_y1, old_y2 = bbox[1].clone(), bbox[4].clone()
            bbox[1] = 1.0 - old_y2
            bbox[4] = 1.0 - old_y1
        if flip_z:
            old_z1, old_z2 = bbox[2].clone(), bbox[5].clone()
            bbox[2] = 1.0 - old_z2
            bbox[5] = 1.0 - old_z1
        bbox = torch.clamp(bbox, 0.0, 1.0)
        has_bbox = torch.tensor(1.0 if bbox.sum() > 0 else 0.0, dtype=torch.float32)
        clinical = torch.tensor(
            self._impute_normalise(s["clinical_raw"], self.clinical_median,
                                   self.clinical_mean, self.clinical_std),
            dtype=torch.float32) if self.clinical_mean is not None else torch.zeros(self.clinical_dim)
        imaging = torch.tensor(
            self._impute_normalise(s["imaging_raw"], self.imaging_median,
                                   self.imaging_mean, self.imaging_std),
            dtype=torch.float32) if self.imaging_mean is not None else torch.zeros(self.imaging_dim)
        return {"volume": volume, "clinical": clinical, "imaging": imaging,
                "bbox_3d": bbox, "has_bbox": has_bbox, "patient_id": s["patient_id"]}

## 3. Model V2 — Fix 2 + Fix 3

**Fix 2 (unchanged):** direct 6-output sigmoid bbox head + DIoU loss.

**Fix 3 (new):** `SliceAttention` now holds a learnable `nn.Embedding(max_depth, feat_dim)`.
Positional vectors are added to slice features once in the model `forward()` and the
same positionally-enriched tensor is fed to both `SliceAttention` and `CrossAttention`.
This gives the model explicit Z-coordinate awareness:
- `SliceAttention` can now weight slices by their anatomical position, not just their content.
- `CrossAttention` (tabular→slice) can now focus on the correct Z-range.
- The bbox head gets Z-aware features, allowing it to predict `z1`/`z2` accurately.

`max_depth=128` means the embedding table is large enough to handle DEPTH=64 or 32
without reloading weights.

In [5]:
class SliceEncoder(nn.Module):
    """Pretrained ResNet18 applied per-slice. Input [B,1,D,H,W] → [B,D,512]."""
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = tv_models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
        orig_w = resnet.conv1.weight.data
        new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        new_conv.weight.data = orig_w.mean(dim=1, keepdim=True)
        resnet.conv1 = new_conv
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 512

    def forward(self, volume):
        B, C, D, H, W = volume.shape
        x = volume.permute(0, 2, 1, 3, 4).contiguous().view(B * D, C, H, W)
        return self.gap(self.backbone(x)).flatten(1).view(B, D, self.feature_dim)


class SliceAttention(nn.Module):
    """
    FIX 3: Learned attention pooling over depth WITH positional encoding.
    Input [B,D,512] — already positionally-encoded by the caller.
    Returns [B,512] pooled feature and [B,D] attention weights.

    The pos_embed table is stored here (owned by this module) but the
    embedding addition is done once in BBox2DSliceAttentionModel.forward()
    so that the same positionally-encoded tensor is shared with CrossAttention.
    """
    def __init__(self, feat_dim=512, hidden=64, max_depth=128):
        super().__init__()
        # FIX 3: learnable positional embedding — one vector per depth position
        self.pos_embed = nn.Embedding(max_depth, feat_dim)
        # Small init: start near v3 behaviour, adapt gradually
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def add_positional_encoding(self, slice_feats):
        """
        Add positional encoding to raw slice features.
        slice_feats: [B, D, F]
        Returns:     [B, D, F]  (same shape, positionally enriched)
        """
        D = slice_feats.shape[1]
        positions = torch.arange(D, device=slice_feats.device)  # [0, 1, ..., D-1]
        return slice_feats + self.pos_embed(positions).unsqueeze(0)  # broadcast over B

    def forward(self, pos_slice_feats):
        """pos_slice_feats: already positionally-encoded [B, D, F]."""
        weights = torch.softmax(self.attn(pos_slice_feats).squeeze(-1), dim=1)  # [B,D]
        pooled  = (weights.unsqueeze(-1) * pos_slice_feats).sum(dim=1)          # [B,F]
        return pooled, weights


class CrossAttention(nn.Module):
    """Tabular query attends to slice features. query [B,Q], context [B,D,K] → [B,Q]."""
    def __init__(self, query_dim, key_dim):
        super().__init__()
        self.proj_q = nn.Linear(query_dim, key_dim)
        self.proj_v = nn.Linear(key_dim, query_dim)
        self.scale  = key_dim ** -0.5
    def forward(self, query, context):
        q = self.proj_q(query).unsqueeze(1)
        w = torch.softmax((q @ context.transpose(1, 2)) * self.scale, dim=-1)
        return query + self.proj_v((w @ context).squeeze(1))


class BBox2DSliceAttentionModel(nn.Module):
    """
    v4: All three bug fixes applied.
    Fix 1 (dataset) — label sync with augmentation.
    Fix 2 — direct 6-output sigmoid bbox head.
    Fix 3 — learnable positional encoding on slice dimension:
        - pos_embed lives in SliceAttention
        - add_positional_encoding() called once in forward()
        - positionally-encoded tensor shared with CrossAttention
          so BOTH attention modules have full Z awareness
    """
    def __init__(self, clinical_dim, imaging_dim, pretrained=True,
                 depth=32, max_depth=128):
        super().__init__()
        self.slice_encoder   = SliceEncoder(pretrained=pretrained)
        # FIX 3: pass max_depth to SliceAttention so pos_embed is large enough
        self.slice_attention = SliceAttention(512, 64, max_depth=max_depth)
        tab_dim = clinical_dim + imaging_dim
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tab_dim, 256), nn.ReLU(True), nn.Dropout(0.15),
            nn.Linear(256, 128),     nn.ReLU(True), nn.Dropout(0.10))
        self.cross_attention = CrossAttention(query_dim=128, key_dim=512)
        self.fusion = nn.Sequential(
            nn.Linear(640, 256), nn.ReLU(True), nn.Dropout(0.15),
            nn.Linear(256, 128), nn.ReLU(True))
        # Fix 2: single direct bbox head — no center/size trick
        self.bbox_head = nn.Linear(128, 6)
        # No bias init override — zero init + DIoU handles early exploration

    def get_optimizer_param_groups(self, backbone_lr=2e-4, head_lr=3e-4, wd=1e-3):
        backbone_params = list(self.slice_encoder.backbone.parameters())
        bp_ids = {id(p) for p in backbone_params}
        other  = [p for p in self.parameters() if id(p) not in bp_ids]
        return [{"params": backbone_params, "lr": backbone_lr, "weight_decay": wd},
                {"params": other,           "lr": head_lr,     "weight_decay": wd}]

    @staticmethod
    def decode_bbox(raw):
        """Fix 2: sorted-sigmoid → [x1,y1,z1,x2,y2,z2] with guaranteed min<=max."""
        sig  = torch.sigmoid(raw)
        mins = torch.min(sig[:, :3], sig[:, 3:])
        maxs = torch.max(sig[:, :3], sig[:, 3:])
        return torch.cat([mins, maxs], dim=1)

    def forward(self, volume, clinical, imaging):
        sf = self.slice_encoder(volume)                                 # [B, D, 512]

        # FIX 3: add positional encoding ONCE, share with both attention modules
        # sf_pos carries Z-position information:
        #   - SliceAttention can weight slices by anatomical position
        #   - CrossAttention can focus the tabular query on the correct Z-range
        sf_pos = self.slice_attention.add_positional_encoding(sf)       # [B, D, 512]

        img_feat, attn_w = self.slice_attention(sf_pos)                 # [B,512], [B,D]
        tab = self.tabular_encoder(torch.cat([clinical, imaging], dim=1))  # [B,128]
        tab = self.cross_attention(tab, sf_pos)                         # [B,128]
        fused = self.fusion(torch.cat([img_feat, tab], dim=1))          # [B,128]
        bbox  = self.decode_bbox(self.bbox_head(fused))
        return {"bbox_3d": bbox, "slice_attn": attn_w}

print("Model v4 defined (Fix 3 — positional encoding applied).")

Model v4 defined (Fix 3 — positional encoding applied).


## 4. Loss Functions and Metrics
Unchanged from v3 — IoU, GIoU, DIoU (Fix 2), center distance.

In [6]:
def bbox_iou_3d(pred, target):
    """3D IoU. [B,6] → [B]"""
    px1,py1,pz1,px2,py2,pz2 = pred[:,0],pred[:,1],pred[:,2],pred[:,3],pred[:,4],pred[:,5]
    tx1,ty1,tz1,tx2,ty2,tz2 = target[:,0],target[:,1],target[:,2],target[:,3],target[:,4],target[:,5]
    iw  = torch.clamp(torch.min(px2,tx2) - torch.max(px1,tx1), min=0)
    ih  = torch.clamp(torch.min(py2,ty2) - torch.max(py1,ty1), min=0)
    id_ = torch.clamp(torch.min(pz2,tz2) - torch.max(pz1,tz1), min=0)
    inter = iw * ih * id_
    pv = torch.clamp(px2-px1,min=0)*torch.clamp(py2-py1,min=0)*torch.clamp(pz2-pz1,min=0)
    tv = torch.clamp(tx2-tx1,min=0)*torch.clamp(ty2-ty1,min=0)*torch.clamp(tz2-tz1,min=0)
    return inter / (pv + tv - inter + 1e-6)


def bbox_giou_3d(pred, target):
    """3D GIoU. [B,6] → [B]"""
    px1,py1,pz1,px2,py2,pz2 = pred[:,0],pred[:,1],pred[:,2],pred[:,3],pred[:,4],pred[:,5]
    tx1,ty1,tz1,tx2,ty2,tz2 = target[:,0],target[:,1],target[:,2],target[:,3],target[:,4],target[:,5]
    iw  = torch.clamp(torch.min(px2,tx2) - torch.max(px1,tx1), min=0)
    ih  = torch.clamp(torch.min(py2,ty2) - torch.max(py1,ty1), min=0)
    id_ = torch.clamp(torch.min(pz2,tz2) - torch.max(pz1,tz1), min=0)
    inter = iw * ih * id_
    pv = torch.clamp(px2-px1,min=0)*torch.clamp(py2-py1,min=0)*torch.clamp(pz2-pz1,min=0)
    tv = torch.clamp(tx2-tx1,min=0)*torch.clamp(ty2-ty1,min=0)*torch.clamp(tz2-tz1,min=0)
    union = pv + tv - inter + 1e-6
    iou   = inter / union
    ew = torch.clamp(torch.max(px2,tx2) - torch.min(px1,tx1), min=0)
    eh = torch.clamp(torch.max(py2,ty2) - torch.min(py1,ty1), min=0)
    ed = torch.clamp(torch.max(pz2,tz2) - torch.min(pz1,tz1), min=0)
    enc_v = ew * eh * ed + 1e-6
    return iou - (enc_v - union) / enc_v


def bbox_diou_3d(pred, target):
    """3D DIoU. [B,6] → [B]. Gradient even at zero overlap — breaks mean-box collapse."""
    iou      = bbox_iou_3d(pred, target)
    pred_c   = (pred[:,   :3] + pred[:,   3:]) / 2.0
    gt_c     = (target[:, :3] + target[:, 3:]) / 2.0
    dist2    = ((pred_c - gt_c) ** 2).sum(dim=1)
    enc_min  = torch.min(pred[:, :3], target[:, :3])
    enc_max  = torch.max(pred[:, 3:], target[:, 3:])
    enc_diag2 = ((enc_max - enc_min) ** 2).sum(dim=1).clamp(min=1e-6)
    return iou - dist2 / enc_diag2


def center_distance_3d(pred, target):
    pred_c   = (pred[:,   :3] + pred[:,   3:]) / 2.0
    gt_c     = (target[:, :3] + target[:, 3:]) / 2.0
    dist_abs = torch.norm(pred_c - gt_c, dim=1)
    gt_diag  = torch.norm(target[:, 3:] - target[:, :3], dim=1).clamp(min=1e-6)
    return dist_abs, dist_abs / gt_diag

print("Loss / metric functions defined.")

Loss / metric functions defined.


## 5. Trainer V4
Changes vs v3:
- `patience` raised 15 → 20 (curves were still moving at patience trigger).
- `ReduceLROnPlateau` removed — it fired too early in v3 and dropped LR before the
  positional encoding had time to learn; cosine decay is sufficient.
- Prints Z-coordinate accuracy in addition to overall center distance so we can
  directly observe Fix 3 working: watch `val_z_dist` drop across epochs.

In [7]:
class BBox3DTrainerV4:
    def __init__(self, model, device,
                 backbone_lr=2e-4, head_lr=3e-4, weight_decay=1e-3,
                 lambda_l1=0.20, lambda_iou=0.25, lambda_giou=0.10,
                 lambda_diou=0.25, lambda_center=0.20,
                 warmup_epochs=5, total_epochs=120):
        self.model         = model
        self.device        = device
        self.lambda_l1     = lambda_l1
        self.lambda_iou    = lambda_iou
        self.lambda_giou   = lambda_giou
        self.lambda_diou   = lambda_diou
        self.lambda_center = lambda_center
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.l1_loss = nn.SmoothL1Loss(reduction="none", beta=0.1)
        if hasattr(model, "get_optimizer_param_groups"):
            pgs = model.get_optimizer_param_groups(backbone_lr, head_lr, weight_decay)
        else:
            pgs = [{"params": model.parameters(), "lr": head_lr,
                    "weight_decay": weight_decay}]
        for pg in pgs:
            pg["initial_lr"] = pg["lr"]
        self.optimizer = optim.AdamW(pgs)
        self.scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
        self.history   = defaultdict(list)
        self.best_iou  = -1.0

    def _lr_scale(self, epoch):
        if epoch < self.warmup_epochs:
            return (epoch + 1) / self.warmup_epochs
        p = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * p))

    def _update_lr(self, epoch):
        s = self._lr_scale(epoch)
        for pg in self.optimizer.param_groups:
            pg["lr"] = pg["initial_lr"] * s

    def _compute_loss(self, pred, gt, has_bbox):
        mask  = has_bbox.float()
        denom = mask.sum().clamp(min=1.0)
        l1    = (self.l1_loss(pred, gt).mean(dim=1) * mask).sum() / denom
        iou   = bbox_iou_3d(pred, gt)
        giou  = bbox_giou_3d(pred, gt)
        diou  = bbox_diou_3d(pred, gt)
        dist_abs, dist_rel = center_distance_3d(pred, gt)
        iou_l  = ((1.0 - iou)  * mask).sum() / denom
        giou_l = ((1.0 - giou) * mask).sum() / denom
        diou_l = ((1.0 - diou) * mask).sum() / denom
        cd_l   = (dist_abs      * mask).sum() / denom
        total  = (self.lambda_l1     * l1
                + self.lambda_iou    * iou_l
                + self.lambda_giou   * giou_l
                + self.lambda_diou   * diou_l
                + self.lambda_center * cd_l)
        # Z-only center distance — key diagnostic for Fix 3
        pred_z_c = (pred[:, 2] + pred[:, 5]) / 2.0
        gt_z_c   = (gt[:,   2] + gt[:,   5]) / 2.0
        z_dist   = (torch.abs(pred_z_c - gt_z_c) * mask).sum() / denom
        return total, iou.detach(), dist_abs.detach(), dist_rel.detach(), z_dist.detach()

    def _run_epoch(self, loader, train=True):
        self.model.train(train)
        total_loss = 0.0
        all_iou = []; all_dist = []; all_drel = []; all_zdist = []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for batch in loader:
                vol  = batch["volume"].to(self.device,   non_blocking=True)
                clin = batch["clinical"].to(self.device, non_blocking=True)
                img  = batch["imaging"].to(self.device,  non_blocking=True)
                gt   = batch["bbox_3d"].to(self.device,  non_blocking=True)
                hb   = batch["has_bbox"].to(self.device, non_blocking=True)
                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(self.device.type == "cuda")):
                    out  = self.model(vol, clin, img)
                    pred = out["bbox_3d"]
                    loss, iou, dist_abs, dist_rel, z_dist = self._compute_loss(pred, gt, hb)
                if train:
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                total_loss += loss.item()
                valid = hb > 0
                if valid.any():
                    all_iou.extend(iou[valid].cpu().tolist())
                    all_dist.extend(dist_abs[valid].cpu().tolist())
                    all_drel.extend(dist_rel[valid].cpu().tolist())
                    all_zdist.extend(z_dist.unsqueeze(0).cpu().tolist())
        return (
            total_loss / len(loader),
            float(np.mean(all_iou))   if all_iou   else 0.0,
            float(np.mean(all_dist))  if all_dist  else 1.0,
            float(np.mean(all_drel))  if all_drel  else 1.0,
            float(np.mean(all_zdist)) if all_zdist else 1.0,
        )

    def train(self, train_loader, val_loader, num_epochs=120, patience=20,
              save_path="best_3d_bbox_model_v4.pth"):
        print("\n" + "="*70)
        print("TRAINING: Duke 3D BBox Detection v4 (positional encoding)")
        print("="*70)
        print("Watch val_z_dist — it should drop as Fix 3 takes effect.")
        print("="*70)
        no_improve = 0
        for epoch in range(num_epochs):
            self._update_lr(epoch)
            tr = self._run_epoch(train_loader, train=True)
            va = self._run_epoch(val_loader,   train=False)
            for key, val in zip(
                ["train_loss","train_iou","train_center_dist",
                 "train_center_dist_rel","train_z_dist"], tr):
                self.history[key].append(val)
            for key, val in zip(
                ["val_loss","val_iou","val_center_dist",
                 "val_center_dist_rel","val_z_dist"], va):
                self.history[key].append(val)
            lr = self.optimizer.param_groups[-1]["lr"]
            print(f"\nEpoch {epoch+1}/{num_epochs}  LR={lr:.2e}")
            print(f"  Train  loss:{tr[0]:.4f}  IoU:{tr[1]:.4f}  "
                  f"CD:{tr[2]:.4f}  ZDist:{tr[4]:.4f}")
            print(f"  Val    loss:{va[0]:.4f}  IoU:{va[1]:.4f}  "
                  f"CD:{va[2]:.4f}  ZDist:{va[4]:.4f}")
            if va[1] > self.best_iou:
                self.best_iou = va[1]
                no_improve = 0
                torch.save({"epoch": epoch,
                            "model_state_dict": self.model.state_dict(),
                            "best_iou": self.best_iou}, save_path)
                print("  ✓ Best model saved!")
            else:
                no_improve += 1
                print(f"  Patience: {no_improve}/{patience}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        ckpt = torch.load(save_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        print(f"\nBest Val IoU: {self.best_iou:.4f}")

print("Trainer V4 defined.")

Trainer V4 defined.


## 6. Visualization
Extended to show Z-distance training curve alongside the others.

In [8]:
def plot_training_curves(history, save_path="results/3d_bbox_training_curves.png"):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    panels = [
        ("train_loss",            "val_loss",            "Loss"),
        ("train_iou",             "val_iou",             "3D IoU"),
        ("train_center_dist",     "val_center_dist",     "Center Distance (abs)"),
        ("train_center_dist_rel", "val_center_dist_rel", "Center Distance (relative)"),
        ("train_z_dist",          "val_z_dist",          "Z-axis Distance (Fix 3 diagnostic)"),
    ]
    for i, ((tk, vk, title), ax) in enumerate(zip(panels, axes.flatten())):
        ax.plot(history[tk], label="Train", linewidth=2)
        ax.plot(history[vk], label="Val",   linewidth=2, linestyle="--")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
        if title.startswith("Z-axis"):
            ax.set_title(title + "\n(should drop as pos-enc learns Z)",
                         fontsize=10, fontweight="bold")
    axes.flatten()[-1].axis('off')  # 6th panel unused
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")


@torch.no_grad()
def evaluate_and_visualize(model, loader, device, num_samples=8,
                            save_path="results/3d_bbox_predictions.png"):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    rows = []; viz_data = []
    for batch in loader:
        vol  = batch["volume"].to(device)
        clin = batch["clinical"].to(device)
        img  = batch["imaging"].to(device)
        gt   = batch["bbox_3d"]
        ids  = batch["patient_id"]
        out  = model(vol, clin, img)
        pred = out["bbox_3d"].cpu()
        iou  = bbox_iou_3d(pred, gt)
        da, dr = center_distance_3d(pred, gt)
        for i in range(len(ids)):
            rows.append(dict(patient=ids[i], iou=iou[i].item(),
                             center_dist=da[i].item(), center_dist_rel=dr[i].item(),
                             pred=pred[i].numpy().round(3).tolist(),
                             gt=gt[i].numpy().round(3).tolist()))
            if len(viz_data) < num_samples:
                viz_data.append(dict(pid=ids[i], vol=batch["volume"][i, 0].numpy(),
                                     gt=gt[i].numpy(), pred=pred[i].numpy(),
                                     iou=iou[i].item(), cd=da[i].item()))
    print(f"\n{'Patient':<22} {'IoU':>6} {'CenterDist':>11} {'RelDist':>9}  Pred → GT")
    print("-" * 100)
    for r in rows:
        print(f"{r['patient']:<22} {r['iou']:6.3f} {r['center_dist']:11.4f} "
              f"{r['center_dist_rel']:9.4f}  {r['pred']} → {r['gt']}")
    mi  = np.mean([r["iou"]             for r in rows])
    md_ = np.mean([r["center_dist"]     for r in rows])
    mr  = np.mean([r["center_dist_rel"] for r in rows])
    print("-" * 100)
    print(f"{'MEAN':<22} {mi:6.3f} {md_:11.4f} {mr:9.4f}")
    n = len(viz_data)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1: axes = axes[np.newaxis, :]
    view_names = ["Axial (XY)", "Coronal (XZ)", "Sagittal (YZ)"]
    for ri, d in enumerate(viz_data):
        vol  = d["vol"]; gt = d["gt"]; pred = d["pred"]
        D, H, W = vol.shape
        gx1,gy1,gz1,gx2,gy2,gz2 = gt[0]*W,gt[1]*H,gt[2]*D,gt[3]*W,gt[4]*H,gt[5]*D
        px1,py1,pz1,px2,py2,pz2 = pred[0]*W,pred[1]*H,pred[2]*D,pred[3]*W,pred[4]*H,pred[5]*D
        z_c = int(np.clip((gz1 + gz2) / 2, 0, D - 1))
        views = [
            (vol[z_c, :, :],  (gx1,gy1,gx2-gx1,gy2-gy1), (px1,py1,px2-px1,py2-py1)),
            (vol[:, H//2, :], (gx1,gz1,gx2-gx1,gz2-gz1), (px1,pz1,px2-px1,pz2-pz1)),
            (vol[:, :, W//2], (gz1,gy1,gz2-gz1,gy2-gy1), (pz1,py1,pz2-pz1,py2-py1)),
        ]
        for ci, (slc, gr, pr) in enumerate(views):
            ax = axes[ri, ci]
            s  = (slc - slc.min()) / (slc.max() - slc.min() + 1e-8)
            ax.imshow(s, cmap="gray", origin="upper")
            for (x, y, w, h), col, ls, lbl in [(gr,"lime","-","GT"),(pr,"red","--","Pred")]:
                if w > 0 and h > 0:
                    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False,
                                 edgecolor=col, linewidth=2, linestyle=ls, label=lbl))
            title = (f"{d['pid']}\n{view_names[ci]}  IoU={d['iou']:.3f} CD={d['cd']:.3f}"
                     if ci == 0 else view_names[ci])
            ax.set_title(title, fontsize=8); ax.axis("off")
            if ci == 0: ax.legend(loc="upper right", fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")
    return rows

print("Visualization functions defined.")

Visualization functions defined.


## 7. Run

In [9]:
# ─── Configuration ────────────────────────────────────────────────────────────
EXPORT_DIR  = "exported_patients"
DEPTH       = 32    # → set to 64 once this run converges (needs ~2× GPU memory)
IMAGE_SIZE  = 128   # → set to 192 alongside DEPTH=64
BATCH_SIZE  = 4     # → reduce to 2 if increasing DEPTH/IMAGE_SIZE
NUM_EPOCHS  = 120
PATIENCE    = 20
PRETRAINED  = True
NUM_WORKERS = 4
MAX_DEPTH   = 128   # pos_embed table size — supports up to DEPTH=128 without reinit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  DEPTH: {DEPTH}  |  IMAGE_SIZE: {IMAGE_SIZE}  |  Batch: {BATCH_SIZE}")

Device: cuda  |  DEPTH: 32  |  IMAGE_SIZE: 128  |  Batch: 4


In [10]:
print("Loading train dataset...")
train_ds = Duke3DBBoxDatasetV3(
    Path(EXPORT_DIR) / "train",
    depth=DEPTH, image_size=IMAGE_SIZE, augment=True)

print("\nLoading val dataset...")
val_ds = Duke3DBBoxDatasetV3(
    Path(EXPORT_DIR) / "test",
    depth=DEPTH, image_size=IMAGE_SIZE, augment=False,
    clinical_median=train_ds.clinical_median,
    clinical_mean=train_ds.clinical_mean,
    clinical_std=train_ds.clinical_std,
    imaging_median=train_ds.imaging_median,
    imaging_mean=train_ds.imaging_mean,
    imaging_std=train_ds.imaging_std)
val_ds.clinical_dim = train_ds.clinical_dim
val_ds.imaging_dim  = train_ds.imaging_dim

print(f"\nTrain: {len(train_ds)}  |  Val: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

Loading train dataset...
Dataset: 727 patients  clinical_dim=14  imaging_dim=13

Loading val dataset...
Dataset: 175 patients  clinical_dim=14  imaging_dim=13

Train: 727  |  Val: 175


In [11]:
# ─── Fix 3 sanity check — verify pos_embed is trainable and correct size ──────
model = BBox2DSliceAttentionModel(
    clinical_dim=train_ds.clinical_dim,
    imaging_dim=train_ds.imaging_dim,
    pretrained=PRETRAINED,
    depth=DEPTH,
    max_depth=MAX_DEPTH).to(device)

pe = model.slice_attention.pos_embed
print(f"pos_embed shape   : {pe.weight.shape}  ← should be [{MAX_DEPTH}, 512]")
print(f"pos_embed trainable: {pe.weight.requires_grad}  ← must be True")
print(f"pos_embed std      : {pe.weight.std().item():.4f}  ← should be ~0.02")

# Verify positional encoding is different per position
with torch.no_grad():
    dummy_vol  = torch.zeros(2, 1, DEPTH, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dummy_clin = torch.zeros(2, train_ds.clinical_dim, device=device)
    dummy_img  = torch.zeros(2, train_ds.imaging_dim,  device=device)
    sf = model.slice_encoder(dummy_vol)
    sf_pos = model.slice_attention.add_positional_encoding(sf)
    pos_diff = (sf_pos[:, 0, :] - sf_pos[:, -1, :]).abs().mean().item()
print(f"Pos diff (slice 0 vs last): {pos_diff:.6f}  ← must be > 0")
print(f"  {'PASS — different positions have different encodings' if pos_diff > 0 else 'FAIL'}")

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
pe_params = pe.weight.numel()
print(f"\nModel: {total:,} params ({trainable:,} trainable)")
print(f"pos_embed adds: {pe_params:,} params ({pe_params/total*100:.2f}% of total)")

pos_embed shape   : torch.Size([128, 512])  ← should be [128, 512]
pos_embed trainable: True  ← must be True
pos_embed std      : 0.0199  ← should be ~0.02
Pos diff (slice 0 vs last): 0.023247  ← must be > 0
  PASS — different positions have different encodings

Model: 11,638,215 params (11,638,215 trainable)
pos_embed adds: 65,536 params (0.56% of total)


In [12]:
save_path = "best_3d_bbox_model_v4.pth"

trainer = BBox3DTrainerV4(
    model=model, device=device,
    backbone_lr=2e-4,
    head_lr=3e-4,
    weight_decay=1e-3,
    lambda_l1=0.20,
    lambda_iou=0.25,
    lambda_giou=0.10,
    lambda_diou=0.25,
    lambda_center=0.20,
    warmup_epochs=5,
    total_epochs=NUM_EPOCHS)

trainer.train(train_loader, val_loader,
              num_epochs=NUM_EPOCHS, patience=PATIENCE, save_path=save_path)


TRAINING: Duke 3D BBox Detection v4 (positional encoding)
Watch val_z_dist — it should drop as Fix 3 takes effect.

Epoch 1/120  LR=6.00e-05
  Train  loss:0.7376  IoU:0.0139  CD:0.2930  ZDist:0.1121
  Val    loss:0.7102  IoU:0.0149  CD:0.2918  ZDist:0.1168
  ✓ Best model saved!

Epoch 2/120  LR=1.20e-04
  Train  loss:0.7103  IoU:0.0166  CD:0.2880  ZDist:0.1146
  Val    loss:0.7079  IoU:0.0163  CD:0.2883  ZDist:0.1162
  ✓ Best model saved!

Epoch 3/120  LR=1.80e-04
  Train  loss:0.7086  IoU:0.0171  CD:0.2863  ZDist:0.1113
  Val    loss:0.7085  IoU:0.0156  CD:0.2880  ZDist:0.1148
  Patience: 1/20

Epoch 4/120  LR=2.40e-04
  Train  loss:0.7079  IoU:0.0177  CD:0.2840  ZDist:0.1129
  Val    loss:0.7079  IoU:0.0159  CD:0.2840  ZDist:0.1135
  Patience: 2/20

Epoch 5/120  LR=3.00e-04
  Train  loss:0.7071  IoU:0.0188  CD:0.2844  ZDist:0.1124
  Val    loss:0.7060  IoU:0.0175  CD:0.2852  ZDist:0.1145
  ✓ Best model saved!

Epoch 6/120  LR=3.00e-04
  Train  loss:0.7062  IoU:0.0191  CD:0.2820  ZDi

In [13]:
plot_training_curves(trainer.history)

results = evaluate_and_visualize(
    model, val_loader, device, num_samples=8,
    save_path="results/3d_bbox_predictions.png")

print(f"\nBest Val IoU : {trainer.best_iou:.4f}")
print(f"Model saved  : {save_path}")
print()
print("Fix 3 worked if:")
print("  - val_z_dist curve drops across epochs (Z-axis being learned)")
print("  - val IoU > 0.20 (Z-component contributing positively)")
print("  - prediction images show boxes on CORRECT Z-range (coronal/sagittal views)")
print()
print("Next step if IoU > 0.25: set DEPTH=64, IMAGE_SIZE=192, BATCH_SIZE=2 and retrain.")

Saved: results/3d_bbox_training_curves.png

Patient                   IoU  CenterDist   RelDist  Pred → GT
----------------------------------------------------------------------------------------------------
Patient_001             0.169      0.0556    0.3054  [0.6859999895095825, 0.4519999921321869, 0.5389999747276306, 0.8190000057220459, 0.5979999899864197, 0.781000018119812] → [0.6880000233650208, 0.5220000147819519, 0.5600000023841858, 0.7609999775886536, 0.6050000190734863, 0.7039999961853027]
Patient_005             0.053      0.1555    0.5050  [0.24400000274181366, 0.37299999594688416, 0.3919999897480011, 0.33899998664855957, 0.4749999940395355, 0.5709999799728394] → [0.30799999833106995, 0.41999998688697815, 0.4779999852180481, 0.3970000147819519, 0.4749999940395355, 0.7670000195503235]
Patient_006             0.000      0.3415    3.0627  [0.6850000023841858, 0.652999997138977, 0.5429999828338623, 0.781000018119812, 0.7620000243186951, 0.7070000171661377] → [0.796999990940094, 